# Chapter 30 — Your Dissertation in Python

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch30_dissertation_in_python.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 30.1 From a topic to a question

## 30.2 A project folder, a data log and a README

In [ ]:
import importlib
import inspect
import shutil
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import mospi_nas

BOOK = Path.cwd()                               # the book folder
PROJECT = Path(tempfile.gettempdir()) / "salaried_jobs"
shutil.rmtree(PROJECT, ignore_errors=True)      # start this demo from empty
for sub in ["data/raw", "data/clean", "code", "output/tables",
            "output/figures", "paper"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)

mospi_nas(25)                                   # make sure the cached file exists
cached = BOOK / ("data/raw/mospi_nas_getNASData_base_year-2011-12_"
                 "frequency_code-1_indicator_code-25_series-Current.csv")
raw_income = PROJECT / "data/raw/mospi_nas_pc_nsdp_code25.csv"
shutil.copy2(cached, raw_income)                # copy2 keeps the file's dates
PLFS_DIR = BOOK / "data/plfs"                   # PLFS stays OUTSIDE the project
fetched = pd.read_csv(raw_income, usecols=["fetched_on"])["fetched_on"].iloc[0]
plfs_date = pd.Timestamp((PLFS_DIR / "perv1.csv").stat().st_mtime, unit="s")

In [ ]:
data_log = pd.DataFrame([
    {"file": "data/raw/mospi_nas_pc_nsdp_code25.csv",
     "dataset": "Per capita NSDP by state, current and 2011-12 prices",
     "publisher": "MoSPI, National Accounts Statistics",
     "source": "https://api.mospi.gov.in (nas/getNASData, indicator_code=25)",
     "accessed": fetched, "terms": "published statistics; cite MoSPI",
     "in_project_repo": "yes"},
    {"file": "PLFS_DIR/perv1.csv and Data_LayoutPLFS_2023-24.xlsx",
     "dataset": "PLFS July 2023-June 2024, person file, first visit",
     "publisher": "NSO, MoSPI",
     "source": "https://microdata.gov.in (registration required)",
     "accessed": str(plfs_date.date()),
     "terms": "no redistribution; no attempt to identify respondents; cite NSO",
     "in_project_repo": "NO"},
])
data_log.to_csv(PROJECT / "data_log.csv", index=False)

readme = """# Do richer Indian states have more salaried jobs?

Every table and figure in `output/` is rebuilt from `data/raw/` and the PLFS
files by one command.

## Data
See `data_log.csv`. The PLFS files are NOT included: register at
https://microdata.gov.in, download PLFS 2023-24 (CSV and data layout) and
put them in `data/plfs/` or set the environment variable PLFS_DIR.

## Rebuild
    python -m venv .venv
    .venv\\Scripts\\activate            (Mac/Linux: source .venv/bin/activate)
    pip install -r requirements.txt
    python code/run_all.py
"""
(PROJECT / "README.md").write_text(readme, encoding="utf-8")
print(data_log[["dataset", "accessed", "in_project_repo"]].to_string(index=False))

## 30.3 Cleaning in functions, in a module

In [ ]:
SRC = BOOK / "projects" / "ch30_salaried"   # salaried.py and run_all.py
RAW = ("https://raw.githubusercontent.com/slxnjdhv/"
       "python-for-indian-economics/main/projects/ch30_salaried/")
for f in ["salaried.py", "run_all.py"]:
    if (SRC / f).exists():
        shutil.copy2(SRC / f, PROJECT / "code" / f)
    else:                                       # e.g. on Colab
        import requests
        reply = requests.get(RAW + f, timeout=60)
        reply.raise_for_status()
        (PROJECT / "code" / f).write_text(reply.text, encoding="utf-8")

sys.path.insert(0, str(PROJECT / "code"))       # let Python find the module
import salaried as s
s = importlib.reload(s)                         # pick up edits on a rerun
print(len((PROJECT / "code/salaried.py").read_text(encoding="utf-8").splitlines()),
      "lines;", ", ".join(n for n, f in inspect.getmembers(s, inspect.isfunction)
                          if f.__module__ == "salaried"))

In [ ]:
print(inspect.getsource(s.load_workers))

## 30.4 First, reproduce a published number

In [ ]:
from indiadata import read_plfs, labour_status, labour_rates

p = read_plfs("perv1", folder=PLFS_DIR, columns=["age", "pas", "sas"])
p["status"] = labour_status(p["pas"], p["sas"])
check = pd.DataFrame({"ours": labour_rates(p[p["age"] >= 15]).round(1),
                      "MoSPI": [60.1, 58.2, 3.2]})
del p
check

In [ ]:
names = s.state_names(PLFS_DIR)
income = s.state_income(raw_income)
income.to_csv(PROJECT / "data/clean/state_income_2023_24.csv", index=False)
workers = s.load_workers(PLFS_DIR)
d = s.add_state(workers, names, income)

dropped = sorted(set(names["state"]) - set(income["state"]))
print(f"Workers aged 15-59 (usual principal status): {len(workers):,}")
print(f"... in the {d['state'].nunique()} states with per capita NSDP: {len(d):,}")
print("No per capita NSDP (code 25):", dropped)
del workers

## 30.5 The descriptive picture

In [ ]:
tab = s.state_table(d)
tab.round(2).to_csv(PROJECT / "output/tables/table1_state_shares.csv")
r = np.corrcoef(np.log(tab["pc_nsdp"]), tab["salaried_pct"])[0, 1]
print(f"States: {len(tab)}; correlation of salaried share with log income: {r:.2f}")
top = tab["salaried_pct"].sort_values(ascending=False)
print("Highest salaried shares (%):", top.head(5).round(1).to_dict())
print("Lowest (%):", top.tail(3).round(1).to_dict())

fig = s.figure_states(tab)
s.save(fig, PROJECT / "output/figures/figure1_states")
plt.show()

## 30.6 Regressions and a publication table

In [ ]:
models = s.fit_models(d)
table = s.regression_table(models)
print(table)

b = pd.Series({k: m.params["log_pc"] for k, m in models.items()})
print(f"Doubling income, model (1): {100 * b['(1)'] * np.log(2):.1f} points more "
      f"likely to be salaried; model (3): {100 * b['(3)'] * np.log(2):.1f} points")
print(f"Share of the raw gradient left after controls: {b['(3)'] / b['(1)']:.0%}")

In [ ]:
out = PROJECT / "output/tables"
t = table.tables[0]                                   # a pandas DataFrame
t.to_csv(out / "table2_regressions.csv")              # opens in Excel and Word
(out / "table2_regressions.tex").write_text(table.as_latex(), encoding="utf-8")
(out / "table2_regressions.md").write_text(s.to_markdown(t), encoding="utf-8")
print(s.to_markdown(t.iloc[:2]))

### Is the result robust?

In [ ]:
d["income_group"] = d["state"].map(pd.qcut(tab["pc_nsdp"], 3,
                                           labels=["Poorest 10", "Middle 10",
                                                   "Richest 10"]))
within = (d.groupby(["edu", "income_group"], observed=True)
           .apply(lambda g: pd.Series({
               "salaried_pct": 100 * np.average(g["salaried"], weights=g["weight"]),
               "sample": len(g)}))
           .unstack("income_group"))
shares = within["salaried_pct"]
gap = shares["Richest 10"] - shares["Poorest 10"]
print("Richest minus poorest group, percentage points:",
      gap.round(1).to_dict())
print(f"Smallest cell: {int(within['sample'].min().min()):,} workers")

fig, ax = plt.subplots(figsize=(9, 4))
shares.plot.bar(ax=ax, color=["#9db7d5", "#1f4e79", "#c55a11"], width=0.8, rot=0)
ax.set_xticks(range(len(shares)),
              labels=[e.replace("/", "/\n") for e in shares.index])
ax.set_ylabel("Salaried, % of workers")
ax.set_xlabel("")
ax.set_title("Within every education level, workers in richer states are "
             "more often salaried", loc="left", fontsize=12)
ax.legend(title="States grouped by per capita NSDP", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.08, "Workers aged 15-59, usual principal status, weighted. "
         "Source: author's calculations from PLFS 2023-24 and MoSPI NAS.",
         fontsize=8, color="grey")
plt.show()

## 30.7 One script to rebuild everything

In [ ]:
run_all = (PROJECT / "code/run_all.py").read_text(encoding="utf-8").splitlines()
print("\n".join(run_all[19:28] + ["..."] + run_all[32:33]))

In [ ]:
import gc
import hashlib
import os
import subprocess


def fingerprints(folder: Path) -> pd.Series:
    """SHA-256 of every file below a folder (first 12 hex digits)."""
    files = sorted(f for f in folder.rglob("*") if f.is_file())
    return pd.Series({str(f.relative_to(folder)):
                      hashlib.sha256(f.read_bytes()).hexdigest()[:12]
                      for f in files})


before = pd.concat([fingerprints(PROJECT / "output"),
                    fingerprints(PROJECT / "data/clean")])
del d, models, table, fig                      # free memory for the new process
plt.close("all")
gc.collect()

shutil.rmtree(PROJECT / "output")              # delete what code made ...
shutil.rmtree(PROJECT / "data/clean")
env = {**os.environ, "PLFS_DIR": str(PLFS_DIR), "MPLBACKEND": "Agg",
       "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}   # low-RAM laptops
run = subprocess.run([sys.executable, "code/run_all.py"], cwd=PROJECT, env=env,
                     capture_output=True, text=True, timeout=600)
print("exit code:", run.returncode, "|", run.stdout.strip(), run.stderr[-500:])

after = pd.concat([fingerprints(PROJECT / "output"),
                   fingerprints(PROJECT / "data/clean")])
same = before.eq(after.reindex(before.index))
print(f"{same.sum()} of {len(before)} files identical after the rebuild "
      f"({len(after)} files now):", ", ".join(before.index))

## 30.8 Environments and Git

In [ ]:
from importlib.metadata import version

pins = {p: version(p) for p in ["pandas", "numpy", "statsmodels",
                                "matplotlib", "openpyxl"]}
requirements = "\n".join(f"{p}=={v}" for p, v in pins.items()) + (
    "\nindiadata @ git+https://github.com/slxnjdhv/python-for-indian-economics.git\n")
(PROJECT / "requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)

In [ ]:
gitignore = """# PLFS microdata: MoSPI terms forbid redistribution
data/plfs/
*.zip
# rebuilt by code/run_all.py
output/
data/clean/
# Python and Jupyter clutter
.venv/
__pycache__/
.ipynb_checkpoints/
"""
(PROJECT / ".gitignore").write_text(gitignore, encoding="utf-8")
print(sorted(f.name for f in PROJECT.iterdir()))

## 30.9 Citing data and software; ethics

In [ ]:
print(f"Analysis in Python {sys.version.split()[0]} with pandas {version('pandas')}, "
      f"NumPy {version('numpy')}, statsmodels {version('statsmodels')} and "
      f"matplotlib {version('matplotlib')}.")

In [ ]:
t1 = pd.read_csv(PROJECT / "output/tables/table1_state_shares.csv",
                 index_col="state")
print(f"Smallest state sample in Table 1: {int(t1['sample'].min())} workers "
      f"({t1['sample'].idxmin()}); states under 1,000:",
      t1.index[t1["sample"] < 1000].tolist())

## 30.10 Using AI assistants responsibly

## 30.11 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading